In [1]:
# ============================================
# STEP 1: Install Required Packages
# ============================================
!pip install -q gradio gradio_pdf
!pip install -q pypdf PyPDF2 pymupdf
!pip install -q sentence-transformers transformers accelerate bitsandbytes
!pip install -q faiss-cpu numpy pandas
!pip install -q pytesseract pillow
!pip install -q opencv-python-headless         # deskew/denoise/binarize scanned pages before OCR
!apt-get -qq install -y tesseract-ocr          # OCR binary for scanned pages

# LlamaIndex packages (open-source only, no Gemini)
!pip install -q llama-index llama-index-readers-file
!pip install -q llama-index-embeddings-huggingface llama-index-vector-stores-faiss

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.6/320.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 114.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 99.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16

## 🔧 Core Imports and Configuration

In [12]:
# ============================================
# STEP 2: Core Imports and Configuration
# ============================================
# Fully open-source stack:
#   - Mistral 7B Instruct (4-bit) via Hugging Face transformers, runs in-notebook
#   - all-MiniLM-L6-v2 for embeddings
# No Gemini, no API keys, nothing cloud-based.
# ============================================
import gradio as gr
from gradio_pdf import PDF
import fitz  # PyMuPDF
from PyPDF2 import PdfReader
import numpy as np
import faiss
import json
import re
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
from datetime import datetime
import hashlib

import shutil
import cv2
import unicodedata

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

# LlamaIndex imports (open-source only)
from llama_index.core import Document, VectorStoreIndex, StorageContext
from llama_index.core.schema import TextNode
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.vector_stores import MetadataFilters, MetadataFilter, FilterOperator

# ---- Local open-source LLM (Mistral 7B, 4-bit, runs on Colab GPU) ----
LLM_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"
# If you hit a gated-model error, switch to this (no license gate):
# LLM_MODEL = "Qwen/Qwen2.5-7B-Instruct"

# If using a gated model, authenticate first (add HF_TOKEN via the Colab key icon):
# from huggingface_hub import login
# from google.colab import userdata
# login(userdata.get("HF_TOKEN"))

_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    quantization_config=_bnb_config,
    device_map="auto",
)

def llm_generate(prompt: str, temperature: float = 0.1, max_new_tokens: int = 512) -> str:
    """Single entry point for all LLM calls. Runs the model in-notebook."""
    messages = [{"role": "user", "content": prompt}]
    _device = next(_model.parameters()).device

    inputs = _tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(_device)
    with torch.no_grad():
        output = _model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=_tokenizer.eos_token_id,
        )
    generated = output[0][inputs["input_ids"].shape[-1]:]
    return _tokenizer.decode(generated, skip_special_tokens=True).strip()

try:
    _test_output = llm_generate("Reply with the single word: OK", max_new_tokens=5)
    print(f"LLM self-test succeeded. Model replied: {_test_output!r}")
except Exception as _e:
    import traceback
    print("=" * 70)
    print(f"LLM SELF-TEST FAILED: {type(_e).__name__}: {_e!r}")
    print("The app will still launch, but every AI-generated answer will fail")
    print("until this is fixed. Common causes:")
    print("  1. mistralai/Mistral-7B-Instruct-v0.2 is a GATED model -- accept")
    print("     the license on its Hugging Face page and authenticate with:")
    print("       from huggingface_hub import login; login('<your HF_TOKEN>')")
    print("     (uncomment the login() lines above), OR switch LLM_MODEL to")
    print("     the ungated 'Qwen/Qwen2.5-7B-Instruct'.")
    print("  2. bitsandbytes 4-bit quantization requires a CUDA GPU -- check")
    print("     Runtime > Change runtime type > GPU if you're on Colab.")
    print("=" * 70)
    traceback.print_exc()

def extract_json(text: str) -> dict:
    """Robustly pull the first JSON object out of an LLM response."""
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if not match:
        return {}
    try:
        return json.loads(match.group())
    except json.JSONDecodeError:
        return {}

# ---- OCR availability check ----
# Tesseract failing silently is hard to diagnose: pytesseract just swallows
# the error and every scanned page comes back as "". Check once, up front,
# and say so loudly instead of leaving it as a mystery three cells later.
_tesseract_path = shutil.which("tesseract")
if _tesseract_path is None:
    print("WARNING: Tesseract OCR binary not found on PATH.")
    print("  Scanned pages will silently extract as empty text.")
    print("  Install it with: !apt-get install -y tesseract-ocr")
else:
    print(f"Tesseract OCR found: {_tesseract_path}")

# Embedding models (both open-source, same model)
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
llama_embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("Imports and configuration complete. Using local Mistral 7B via transformers.")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

LLM self-test succeeded. Model replied: 'Understood. I will'
Tesseract OCR found: /usr/bin/tesseract


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Imports and configuration complete. Using local Mistral 7B via transformers.


## 📄 Data Structures for Enhanced Document Management
Let's define our data structures to handle complex document metadata:

In [3]:
# ============================================
# STEP 3: Data Structures for Document Management
# ============================================
#
# WHAT WE'RE DOING:
# Defining three Python dataclasses that act as structured containers for
# the information we extract from the PDF: individual page data, logical
# document groupings, and rich chunk metadata.
#
# WHY THIS MATTERS:
# A pharmaceutical blob PDF may contain many distinct documents -- a Cover
# Letter, a Certificate of Quality, a Packaging Specification, etc. -- all
# merged into one file. These dataclasses let us track which pages belong
# to which document and carry that metadata through the entire pipeline so
# answers can cite exact page ranges and document types.
#
# WHAT YOU'LL SEE:
# No output. The classes are defined silently and will be used in later
# cells when we process the uploaded PDF.
# ============================================

@dataclass
class PageInfo:
    """Stores information about a single page"""
    page_num: int
    text: str
    doc_type: Optional[str] = None
    page_in_doc: int = 0
    was_ocr: bool = False                       # True if this page went through OCR
    ocr_confidence: Optional[float] = None       # Tesseract mean word confidence (0-100), OCR pages only

@dataclass
class LogicalDocument:
    """Represents a logical document within a PDF"""
    doc_id: str
    doc_type: str
    page_start: int
    page_end: int
    text: str
    chunks: List[Dict] = None

@dataclass
class ChunkMetadata:
    """Rich metadata for each chunk"""
    chunk_id: str
    doc_id: str
    doc_type: str
    chunk_index: int
    page_start: int
    page_end: int
    text: str
    embedding: Optional[np.ndarray] = None

## 🧠 Document Intelligence Functions
These functions handle document classification and boundary detection:

In [4]:
# ============================================
# STEP 4: Document Intelligence Functions
# ============================================
VALID_DOC_TYPES = [
    "Cover Letter", "Certificate Of Quality", "Packaging Specification",
    "BSE/TSE Declaration", "Material Description", "Supplier Qualification",
    "Chain Of Custody", "Other"
]

# ------------------------------------------------------------------
# BUG FIX: the original version relied 100% on the local LLM
# (classify_document_type / detect_document_boundary), and both
# functions swallowed every exception and silently fell back to
# 'Other' / "same document". If the LLM never loaded successfully
# (no GPU, gated model needing an HF token, OOM, etc.) EVERY page
# would silently get classified as 'Other' and EVERY boundary check
# would silently default to "same document" -- which is exactly the
# symptom in the screenshot: "Documents Found: 1", "Types: Other".
# The pipeline reported success because the exception never
# propagated up to process_pdf().
#
# Fix: add a fast, deterministic, LLM-free heuristic classifier based
# on the distinctive titles/headers pharma documents like this always
# carry. This is now the PRIMARY classifier (accurate for structured
# documents like COAs, packaging specs, BSE/TSE declarations, etc.
# and works with zero dependency on the LLM loading correctly). The
# LLM is kept only as an optional secondary refinement for pages the
# heuristic can't confidently label, and any LLM error is now surfaced
# instead of hidden.
# ------------------------------------------------------------------

# Ordered (checked top to bottom, first match wins) title/keyword
# patterns matched against the START of a page's text.
TITLE_PATTERNS = [
    ("Cover Letter", re.compile(
        r"To Whom It May Concern|^\s*Dear\s|^\s*Re:\s|Sincerely,", re.I)),
    ("Certificate Of Quality", re.compile(
        r"Certificate of Quality", re.I)),
    ("Packaging Specification", re.compile(
        r"Packaging (Component )?Specification|PKG-SPEC", re.I)),
    ("BSE/TSE Declaration", re.compile(
        r"Transmissible Spongiform|BSE/TSE|Spongiform Encephalopath", re.I)),
    ("Material Description", re.compile(
        r"Material Description Sheet", re.I)),
    ("Supplier Qualification", re.compile(
        r"Supplier Qualification Record", re.I)),
    ("Chain Of Custody", re.compile(
        r"Chain of Custody", re.I)),
]

# Signals that a page CONTINUES the previous page rather than starting
# a new document (e.g. "(continued)", "Page 2 of 2").
CONTINUATION_PATTERN = re.compile(
    r"\(continued\)|continued\)|\bPage\s+(\d+)\s+of\s+(\d+)", re.I)

def _is_continuation_page(text: str, sample_len: int = 300) -> bool:
    """Heuristic: does this page explicitly mark itself as a continuation?"""
    sample = text[:sample_len]
    m = re.search(r"\bPage\s+(\d+)\s+of\s+(\d+)", sample, re.I)
    if m and int(m.group(1)) > 1:
        return True
    if re.search(r"\(continued\)", sample, re.I):
        return True
    return False

def classify_document_type_heuristic(text: str, sample_len: int = 400) -> str:
    """Fast, deterministic classification using known document headers."""
    sample = text[:sample_len]
    for doc_type, pattern in TITLE_PATTERNS:
        if pattern.search(sample):
            return doc_type
    return "Other"

def clean_doc_type(response):
    """Clean up LLM response to extract a valid doc_type label."""
    cleaned = response.strip().replace('"', '').replace('`', '').replace('*', '').lower().replace(".", "").strip()
    cleaned_title = cleaned.title()
    for label in VALID_DOC_TYPES:
        if label.lower() in cleaned.lower():
            return label
    return cleaned_title

def classify_document_type(text: str, max_length: int = 1500) -> str:
    """
    Classify the document type. Tries the fast heuristic first (reliable
    for structured pharma documents and has no external dependency);
    only falls back to the local LLM when the heuristic can't confidently
    label the page. LLM errors are now logged with the real traceback
    instead of being silently swallowed.
    """
    heuristic_type = classify_document_type_heuristic(text)
    if heuristic_type != "Other":
        return heuristic_type

    text_sample = text[:max_length] if len(text) > max_length else text
    prompt = f"""[INST] You are a pharmaceutical document classifier.
Classify the page into EXACTLY ONE of these types: {VALID_DOC_TYPES}

Definitions (check in this order):
- "Cover Letter": a LETTER. Look for "To Whom It May Concern", "Dear", "Re:",
  "Sincerely", or a signature block. This ALWAYS wins if the page is a letter,
  even if it mentions quality or part numbers.
- "Certificate of Quality": has Lot Number, Date of Manufacture, Expiration Date,
  and a test-results table (Autoclave, Gamma Irradiation, "Conforms").
- "Packaging Specification": packaging components, blister tray, lid film, carton,
  a PKG-SPEC document number.
- "BSE/TSE Declaration": declaration about animal-origin materials / TSE compliance.
- "Material Description": Materials of Construction table, sterilization compatibility,
  physical properties (dimensions, weight).
- "Supplier Qualification": Supplier Name/Code, audit history, ISO 9001/13485
  certifications, approved product list.
- "Chain of Custody": "Chain of Custody", list of assemblies, traceability flow.
- "Other": only if none clearly apply.

Page Content:
{text_sample[:2000]}

OUTPUT: Respond with ONLY the exact type name from the list. [/INST]"""
    try:
        return clean_doc_type(llm_generate(prompt))
    except Exception as e:
        import traceback
        print(f"Classification error (LLM unavailable, keeping heuristic result 'Other'): {e}")
        traceback.print_exc()
        return "Other"

def detect_document_boundary(prev_text: str, curr_text: str,
                            current_doc_type: str = None) -> bool:
    """
    Detect if two consecutive pages belong to the same document.
    Returns True if SAME document, False if a NEW document starts here.

    Heuristic-first (see note above classify_document_type_heuristic):
      1. If the current page explicitly marks itself as a continuation
         ("(continued)", "Page 2 of 2") -> same document.
      2. Else if the current page opens with a recognizable document
         title/header -> new document (even if the title matches the
         previous type, e.g. two back-to-back single-page Certificates
         of Quality for different lots are still two separate documents).
      3. Otherwise, fall back to the LLM for a judgment call; if the LLM
         is unavailable, default to "same document" only as a last resort
         and log the real error.
    """
    if not prev_text or not curr_text:
        return False

    if _is_continuation_page(curr_text):
        return True

    curr_heuristic_type = classify_document_type_heuristic(curr_text)
    if curr_heuristic_type != "Other":
        return False  # fresh, recognizable header -> new document

    prev_sample = prev_text[-500:] if len(prev_text) > 500 else prev_text
    curr_sample = curr_text[:500] if len(curr_text) > 500 else curr_text
    prompt = f"""Determine if these two pages are from the SAME pharmaceutical document.
Current document type: {current_doc_type or 'Unknown'}

A NEW document starts when the page has:
- A different document title or heading (e.g., "Certificate of Quality"
  vs "Packaging Specification" vs "Material Description Sheet")
- A completely different topic or subject matter
- Its own header with a new document number or reference

Pages belong to the SAME document when:
- The second page says "continued" or "page 2 of 2"
- The content directly continues the previous page's discussion
- They share the same document number or title

End of Previous Page:
...{prev_sample}

Start of Current Page:
{curr_sample}...

Answer ONLY 'Yes' if same document or 'No' if different document."""
    try:
        return llm_generate(prompt).lower().startswith('yes')
    except Exception as e:
        import traceback
        print(f"Boundary detection error (LLM unavailable, defaulting to 'same document'): {e}")
        traceback.print_exc()
        return True

## 📑 Advanced PDF Processing Pipeline
Now let's build the enhanced PDF processing pipeline:

In [13]:
# ============================================
# STEP 5: Advanced PDF Processing Pipeline
# ============================================
#
# WHAT WE'RE DOING:
# Defining extract_and_analyze_pdf(), which opens the uploaded PDF page by
# page, extracts text (with a full OCR preprocessing pipeline for scanned
# pages), calls classify_document_type() on the first page of each segment,
# and uses detect_document_boundary() to decide where one pharmaceutical
# document ends and the next begins.
#
# WHY THIS MATTERS:
# A pharmaceutical blob PDF is not a single document -- it is many
# documents concatenated together. This function produces two outputs:
# (1) a list of PageInfo objects (one per page) and (2) a list of
# LogicalDocument objects (one per identified sub-document). Accurate
# boundary detection is critical: if pages are grouped incorrectly,
# retrieved chunks will mix content from unrelated documents.
#
# OCR ROBUSTNESS:
# Real-world scans are rarely clean -- they're rotated a few degrees,
# have uneven lighting, and pick up speckle noise from the scanner or
# fax. Feeding a raw scan straight into Tesseract produces noticeably
# worse text than feeding it a cleaned-up version. Before OCR, each
# scanned page goes through:
#   1. Higher-resolution rendering (2x zoom instead of default ~72 DPI)
#   2. Grayscale conversion
#   3. Denoising (remove scanner/JPEG speckle)
#   4. Deskew (auto-detect and correct rotation)
#   5. Contrast enhancement (CLAHE)
#   6. Binarization (Otsu thresholding -> clean black/white text)
# Tesseract's own per-word confidence scores are then used to flag pages
# where OCR likely failed, rather than silently trusting whatever text
# came out.
#
# TEXT CLEANING:
# Two more passes run on the extracted text itself (separate from the
# image preprocessing above, and applied to native and OCR'd text
# alike):
#   - clean_extracted_text(): per-page, general whitespace/encoding
#     cleanup and OCR-speckle-line removal, applied before
#     classification/boundary detection so those still work correctly.
#   - strip_repeated_boilerplate(): applied once per logical document,
#     after boundaries are already decided, to drop repeated
#     page-number footers/headers before chunking -- content that adds
#     nothing to retrieval but dilutes every chunk's embedding.
#
# WHAT YOU'LL SEE:
# No output when this cell runs. It is called inside process_pdf() in the
# EnhancedDocumentStore class (Step 8) when the user uploads a file.
# ============================================

# Below this length, page text is treated as too unreliable to classify
# or boundary-check with any confidence (empty extraction, failed OCR,
# or a near-blank page).
MIN_TEXT_LENGTH = 20

# Below this mean Tesseract word-confidence (0-100), an OCR'd page is
# flagged as low-confidence rather than trusted outright.
MIN_OCR_CONFIDENCE = 60


def clean_extracted_text(text: str) -> str:
    """General-purpose cleanup applied to every page's text, native or
    OCR'd, before it's classified, boundary-checked, or chunked.

    Deliberately conservative: fixes whitespace/encoding noise without
    touching content words or structural markers like "Page 2 of 2",
    since detect_document_boundary() still needs those intact.
      - Unicode NFKC normalization (e.g. the "ﬁ" ligature -> "fi",
        which OCR and some PDF fonts produce and which would otherwise
        silently break substring/keyword matches)
      - Strip control characters and the "�" replacement character that
        show up from encoding issues or garbled OCR
      - Collapse runs of spaces/tabs to a single space
      - Collapse 3+ blank lines down to 2 (paragraph break)
      - Drop lines that are pure OCR speckle: no letters or digits at all
    """
    if not text:
        return text

    text = unicodedata.normalize("NFKC", text)

    def _is_junk_char(ch: str) -> bool:
        if ch in ("\n", "\t"):
            return False
        if ch == "\ufffd":
            return True
        # NOTE: do not use str.isprintable() here -- it returns False for
        # '\t' itself, which would strip tabs before the whitespace-
        # collapsing regex below ever saw them (collapsing e.g.
        # "Lot\t\tNumber" straight into "LotNumber" with no space at all).
        # Checking the Unicode category directly avoids that trap while
        # still stripping real control/format junk like form feeds.
        return unicodedata.category(ch).startswith("C")

    text = "".join(ch for ch in text if not _is_junk_char(ch))

    # Collapse horizontal whitespace runs, but keep line breaks meaningful.
    lines = [re.sub(r"[ \t]+", " ", line).strip() for line in text.split("\n")]

    # Drop speckle lines: short, and containing no letters/digits at all
    # (e.g. a lone "|" or "." picked up from a scanner artifact).
    lines = [
        line for line in lines
        if line == "" or len(line) > 3 or re.search(r"[A-Za-z0-9]", line)
    ]

    cleaned = "\n".join(lines)
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)  # collapse 3+ blank lines to 1
    return cleaned.strip()


def strip_repeated_boilerplate(text: str) -> str:
    """Remove page-footer/header boilerplate from a *joined, multi-page*
    logical document, right before it's chunked and embedded.

    Only called after boundary detection has already run -- patterns
    like "Page 2 of 2" are exactly what detect_document_boundary() looks
    for, so they must survive on the per-page text. Once boundaries are
    decided, though, these lines are pure noise for retrieval: they add
    no semantic content and just dilute the embedding for every chunk
    that contains one.
    """
    if not text:
        return text

    patterns = [
        r"^\s*Page\s+\d+\s+of\s+\d+\s*$",   # "Page 2 of 2"
        r"^\s*Page\s+\d+\s*$",              # "Page 2"
        r"^\s*-\s*\d+\s*-\s*$",             # "- 2 -"
        r"^\s*\d+\s*$",                     # bare page number on its own line
    ]
    combined = re.compile("|".join(patterns), re.IGNORECASE)

    lines = [line for line in text.split("\n") if not combined.match(line)]
    cleaned = "\n".join(lines)
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)
    return cleaned.strip()


def _deskew_grayscale(gray_img: np.ndarray) -> Tuple[np.ndarray, float]:
    """Detect and correct rotation in a grayscale scanned-page image.

    Uses Otsu thresholding to isolate ink pixels, then fits a minimum-area
    rectangle around them to estimate the skew angle, and rotates the
    image to compensate. Returns the (possibly) rotated image and the
    detected angle in degrees.
    """
    thresh = cv2.threshold(gray_img, 0, 255, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)[1]
    coords = np.column_stack(np.where(thresh > 0))

    if coords.shape[0] < 20:
        # Not enough ink pixels to estimate an angle reliably (e.g. a
        # near-blank page) -- skip deskewing rather than guess.
        return gray_img, 0.0

    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle

    # Don't bother rotating for negligible skew -- avoids introducing
    # interpolation blur on pages that are already straight.
    if abs(angle) < 0.5:
        return gray_img, angle

    (h, w) = gray_img.shape[:2]
    center = (w // 2, h // 2)
    rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(
        gray_img, rotation_matrix, (w, h),
        flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE
    )
    return rotated, angle


def preprocess_scanned_page(pil_img) -> Tuple["Image.Image", float]:
    """Clean up a rendered scanned page before OCR.

    Pipeline: grayscale -> denoise -> deskew -> contrast (CLAHE) ->
    binarize (Otsu). Returns a PIL image ready for pytesseract, plus the
    detected skew angle (useful for logging/debugging OCR quality).
    """
    from PIL import Image

    gray = np.array(pil_img.convert("L"))

    # Denoise: removes scanner/JPEG speckle without blurring text edges
    # as much as a plain Gaussian blur would.
    gray = cv2.fastNlMeansDenoising(gray, h=10)

    # Deskew: rotated scans confuse Tesseract's line segmentation badly.
    gray, skew_angle = _deskew_grayscale(gray)

    # Contrast: CLAHE (adaptive histogram equalization) evens out
    # lighting across the page better than a global contrast stretch,
    # which matters for scans with shadows or uneven scanner lighting.
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)

    # Binarize: Otsu picks the threshold automatically per-page, giving
    # Tesseract clean black text on white background regardless of the
    # original scan's exposure.
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)

    return Image.fromarray(binary), skew_angle


def ocr_with_confidence(pil_img) -> Tuple[str, float]:
    """Run Tesseract and return both the extracted text and its mean
    per-word confidence (0-100), instead of blindly trusting the output."""
    import pytesseract

    data = pytesseract.image_to_data(pil_img, output_type=pytesseract.Output.DICT)
    words, confidences = [], []
    for word, conf in zip(data["text"], data["conf"]):
        if word.strip():
            words.append(word)
            conf_val = int(conf) if str(conf).lstrip("-").isdigit() else -1
            if conf_val != -1:
                confidences.append(conf_val)

    text = " ".join(words)
    mean_confidence = sum(confidences) / len(confidences) if confidences else 0.0
    return text, mean_confidence


def extract_and_analyze_pdf(pdf_file) -> Tuple[List[PageInfo], List[LogicalDocument]]:
    """
    Extract text from PDF and perform intelligent document analysis.
    Returns both page-level info and logical document groupings.
    Supports various file types including scanned PDFs with OCR.
    """
    print("Starting PDF extraction and analysis...")

    # Extract text from each page
    if isinstance(pdf_file, dict) and "content" in pdf_file:
        doc = fitz.open(stream=pdf_file["content"], filetype="pdf")
    elif hasattr(pdf_file, "read"):
        doc = fitz.open(stream=pdf_file.read(), filetype="pdf")
    else:
        doc = fitz.open(pdf_file)

    pages_info = []
    for i, page in enumerate(doc):
        text = page.get_text()
        was_ocr = False
        ocr_confidence = None

        # If no text found, try OCR (for scanned documents)
        if not text.strip():
            print(f"  Page {i}: No text found, attempting OCR...")
            try:
                from PIL import Image
                import io

                # Render at 2x zoom (~144 DPI instead of ~72 DPI) -- the
                # single biggest lever for OCR accuracy before any other
                # preprocessing even runs.
                pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
                img = Image.open(io.BytesIO(pix.tobytes("png")))

                cleaned_img, skew_angle = preprocess_scanned_page(img)
                text, ocr_confidence = ocr_with_confidence(cleaned_img)
                was_ocr = True

                conf_note = f", confidence {ocr_confidence:.0f}%"
                skew_note = f", corrected {skew_angle:.1f}\u00b0 skew" if abs(skew_angle) >= 0.5 else ""
                print(f"  Page {i}: OCR extracted {len(text)} characters{conf_note}{skew_note}")

                if ocr_confidence < MIN_OCR_CONFIDENCE:
                    print(f"  Page {i}: LOW OCR CONFIDENCE ({ocr_confidence:.0f}%) -- "
                          f"text may be unreliable, consider manual review")
            except Exception as e:
                print(f"  Page {i}: OCR failed - {e}")
                text = ""

        # General text cleaning -- whitespace/encoding noise, OCR
        # speckle lines -- applied identically to native and OCR'd text,
        # before it's used for classification, boundary detection, or
        # chunking. Deliberately leaves structural markers (like "Page 2
        # of 2") intact; see strip_repeated_boilerplate() for those.
        text = clean_extracted_text(text)

        pages_info.append(PageInfo(
            page_num=i, text=text, was_ocr=was_ocr, ocr_confidence=ocr_confidence
        ))

    doc.close()

    if not pages_info:
        raise ValueError("No text could be extracted from PDF")

    print(f"Extracted {len(pages_info)} pages")

    # Perform document classification and boundary detection
    print("Analyzing document structure...")
    logical_docs = []
    current_doc_type = None
    current_doc_pages = []
    doc_counter = 0

    for i, page_info in enumerate(pages_info):
        page_text_reliable = len(page_info.text.strip()) >= MIN_TEXT_LENGTH

        if i == 0:
            # First page - classify document type. An unreliable first
            # page (empty extraction, failed/low-confidence OCR) can't
            # be trusted to seed a document type -- label it explicitly
            # rather than feeding near-empty text into the classifier
            # and getting a confident-looking but meaningless guess.
            if page_text_reliable:
                current_doc_type = classify_document_type(page_info.text)
            else:
                current_doc_type = "Unclassified (extraction failed)"
                print(f"  Page {i}: Text too short/unreliable to classify "
                      f"({len(page_info.text.strip())} chars) - flagged as {current_doc_type}")
            page_info.doc_type = current_doc_type
            page_info.page_in_doc = 0
            current_doc_pages = [page_info]
            print(f"  Page {i}: New document detected - {current_doc_type}")
        else:
            # Check if this page continues the previous document. If
            # either page's text is unreliable, default to "same
            # document" rather than letting boundary detection guess off
            # empty/garbage text -- worst case this merges two documents
            # (still searchable together); the alternative (fragmenting
            # one document into spurious pieces with made-up types) is
            # worse and harder to spot downstream.
            if not page_text_reliable or len(pages_info[i - 1].text.strip()) < MIN_TEXT_LENGTH:
                is_same = True
            else:
                prev_text = pages_info[i - 1].text
                is_same = detect_document_boundary(prev_text, page_info.text, current_doc_type)

            if is_same:
                # Continue current document
                page_info.doc_type = current_doc_type
                page_info.page_in_doc = len(current_doc_pages)
                current_doc_pages.append(page_info)
            else:
                # New document detected - save previous and start new
                logical_doc = LogicalDocument(
                    doc_id=f"doc_{doc_counter}",
                    doc_type=current_doc_type,
                    page_start=current_doc_pages[0].page_num,
                    page_end=current_doc_pages[-1].page_num,
                    text=strip_repeated_boilerplate("\n\n".join([p.text for p in current_doc_pages]))
                )
                logical_docs.append(logical_doc)
                doc_counter += 1

                # Start new document
                if page_text_reliable:
                    current_doc_type = classify_document_type(page_info.text)
                else:
                    current_doc_type = "Unclassified (extraction failed)"
                    print(f"  Page {i}: Text too short/unreliable to classify "
                          f"({len(page_info.text.strip())} chars) - flagged as {current_doc_type}")
                page_info.doc_type = current_doc_type
                page_info.page_in_doc = 0
                current_doc_pages = [page_info]
                print(f"  Page {i}: New document detected - {current_doc_type}")

    # Don't forget the last document
    if current_doc_pages:
        logical_doc = LogicalDocument(
            doc_id=f"doc_{doc_counter}",
            doc_type=current_doc_type,
            page_start=current_doc_pages[0].page_num,
            page_end=current_doc_pages[-1].page_num,
            text=strip_repeated_boilerplate("\n\n".join([p.text for p in current_doc_pages]))
        )
        logical_docs.append(logical_doc)

    print(f"Identified {len(logical_docs)} logical documents")
    for ld in logical_docs:
        print(f"   - {ld.doc_type}: Pages {ld.page_start}-{ld.page_end}")

    return pages_info, logical_docs

## ✂️ Intelligent Chunking with Metadata Preservation
We'll provide two chunking approaches - our custom implementation and LlamaIndex's built-in capabilities:

In [6]:
# ============================================
# STEP 6: Intelligent Chunking with Metadata Preservation
# ============================================

def chunk_document_with_metadata(logical_doc: LogicalDocument,
                                chunk_size: int = 100,
                                overlap: int = 20) -> List[ChunkMetadata]:
    """
    Chunk a logical document while preserving rich metadata.
    Uses sliding window with overlap for better context.
    """
    chunks_metadata = []
    words = logical_doc.text.split()

    if len(words) <= chunk_size:
        chunk_meta = ChunkMetadata(
            chunk_id=f"{logical_doc.doc_id}_chunk_0",
            doc_id=logical_doc.doc_id,
            doc_type=logical_doc.doc_type,
            chunk_index=0,
            page_start=logical_doc.page_start,
            page_end=logical_doc.page_end,
            text=logical_doc.text
        )
        chunks_metadata.append(chunk_meta)
    else:
        stride = chunk_size - overlap
        for i, start_idx in enumerate(range(0, len(words), stride)):
            end_idx = min(start_idx + chunk_size, len(words))
            chunk_text = ' '.join(words[start_idx:end_idx])

            chunk_position = start_idx / len(words)
            page_range = logical_doc.page_end - logical_doc.page_start
            relative_page = int(chunk_position * page_range)
            chunk_page_start = logical_doc.page_start + relative_page
            chunk_page_end = min(chunk_page_start + 1, logical_doc.page_end)

            chunk_meta = ChunkMetadata(
                chunk_id=f"{logical_doc.doc_id}_chunk_{i}",
                doc_id=logical_doc.doc_id,
                doc_type=logical_doc.doc_type,
                chunk_index=i,
                page_start=chunk_page_start,
                page_end=chunk_page_end,
                text=chunk_text
            )
            chunks_metadata.append(chunk_meta)

            if end_idx >= len(words):
                break

    return chunks_metadata

def chunk_with_llama_index(logical_doc: LogicalDocument,
                           chunk_size: int = 100,
                           chunk_overlap: int = 20) -> List[Document]:
    """
    Alternative: Use LlamaIndex's advanced chunking with metadata.
    """
    doc = Document(
        text=logical_doc.text,
        metadata={
            "doc_id": logical_doc.doc_id,
            "doc_type": logical_doc.doc_type,
            "page_start": logical_doc.page_start,
            "page_end": logical_doc.page_end,
            "source": f"{logical_doc.doc_type}_document"
        }
    )

    splitter = SentenceSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        paragraph_separator="\n\n",
        separator=" ",
    )

    nodes = splitter.get_nodes_from_documents([doc])

    chunks_metadata = []
    for i, node in enumerate(nodes):
        chunk_meta = ChunkMetadata(
            chunk_id=f"{logical_doc.doc_id}_chunk_{i}",
            doc_id=logical_doc.doc_id,
            doc_type=logical_doc.doc_type,
            chunk_index=i,
            page_start=node.metadata.get("page_start", logical_doc.page_start),
            page_end=node.metadata.get("page_end", logical_doc.page_end),
            text=node.text
        )
        chunks_metadata.append(chunk_meta)

    return chunks_metadata

def process_all_documents(logical_docs: List[LogicalDocument],
                         use_llama_index: bool = False) -> List[ChunkMetadata]:
    """
    Process all logical documents into chunks with metadata.
    """
    all_chunks = []

    for logical_doc in logical_docs:
        if use_llama_index:
            chunks = chunk_with_llama_index(logical_doc)
        else:
            chunks = chunk_document_with_metadata(logical_doc)

        logical_doc.chunks = chunks
        all_chunks.extend(chunks)
        print(f"  {logical_doc.doc_type}: Created {len(chunks)} chunks")

    return all_chunks

## 🎯 Query Routing and Intelligent Retrieval

In [7]:
# ============================================
# STEP 7: Query Routing and Intelligent Retrieval
# ============================================

def predict_query_document_type(query: str) -> Tuple[str, float]:
    """Predict which pharmaceutical document type likely holds the answer."""
    prompt = f"""Analyze this query and predict which pharmaceutical document type
would most likely contain the answer.

Query: "{query}"

Choose the MOST LIKELY type from:
- Cover Letter: Formal letters about product information or storage conditions
- Certificate Of Quality: Lot numbers, manufacture/expiration dates, test results
- Packaging Specification: Packaging components, materials, part numbers
- BSE/TSE Declaration: Animal-origin material declarations, TSE compliance
- Material Description: Materials of construction, sterilization compatibility
- Supplier Qualification: Supplier audits, ISO certifications, approved products
- Chain Of Custody: Manufactured assemblies, traceability, shipment flow
- Other: General or unclear queries

Respond in JSON format:
{{"type": "DocumentType", "confidence": 0.85}}
Confidence should be between 0.0 and 1.0"""
    try:
        result = extract_json(llm_generate(prompt))
        predicted = result.get("type", "Other")
        confidence = float(result.get("confidence", 0.5))
        return clean_doc_type(predicted), confidence
    except Exception as e:
        print(f"Query routing error: {e}")
        return "Other", 0.0

class IntelligentRetriever:
    """
    Advanced retrieval system with metadata filtering and query routing.
    """

    def __init__(self):
        self.index = None
        self.chunks_metadata = []
        self.doc_type_indices = {}

    def build_indices(self, chunks_metadata: List[ChunkMetadata]):
        """
        Build FAISS indices with document type segregation.
        """
        print("Building vector indices...")
        self.chunks_metadata = chunks_metadata

        texts = [chunk.text for chunk in chunks_metadata]
        embeddings = embed_model.encode(texts, show_progress_bar=True)

        for i, chunk in enumerate(chunks_metadata):
            chunk.embedding = embeddings[i]

        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatL2(dim)
        self.index.add(embeddings)

        doc_types = set(chunk.doc_type for chunk in chunks_metadata)
        for doc_type in doc_types:
            type_indices = [i for i, chunk in enumerate(chunks_metadata)
                          if chunk.doc_type == doc_type]
            if type_indices:
                type_embeddings = embeddings[type_indices]
                type_index = faiss.IndexFlatL2(dim)
                type_index.add(type_embeddings)
                self.doc_type_indices[doc_type] = {
                    'index': type_index,
                    'mapping': type_indices
                }

        print(f"Indexed {len(chunks_metadata)} chunks across {len(doc_types)} document types")

    def retrieve(self, query: str, k: int = 4,
                filter_doc_type: Optional[str] = None,
                auto_route: bool = True) -> List[Tuple[ChunkMetadata, float]]:
        """
        Retrieve relevant chunks with optional filtering and routing.
        Returns chunks with relevance scores.
        """
        query_embedding = embed_model.encode([query])

        if filter_doc_type and filter_doc_type in self.doc_type_indices:
            type_data = self.doc_type_indices[filter_doc_type]
            D, I = type_data['index'].search(query_embedding, k)
            chunk_indices = [type_data['mapping'][i] for i in I[0]]
            distances = D[0]
        elif auto_route:
            predicted_type, confidence = predict_query_document_type(query)
            print(f"Query routed to: {predicted_type} (confidence: {confidence:.2f})")

            if confidence > 0.7 and predicted_type in self.doc_type_indices:
                type_data = self.doc_type_indices[predicted_type]
                D, I = type_data['index'].search(query_embedding, k)
                chunk_indices = [type_data['mapping'][i] for i in I[0]]
                distances = D[0]
            else:
                D, I = self.index.search(query_embedding, k)
                chunk_indices = I[0]
                distances = D[0]
        else:
            D, I = self.index.search(query_embedding, k)
            chunk_indices = I[0]
            distances = D[0]

        scores = [max(0.0, 1 - (d / 2)) for d in distances]

        results = [(self.chunks_metadata[i], scores[idx])
                  for idx, i in enumerate(chunk_indices)]

        return results

## 💬 Enhanced Answer Generation with Source Attribution

In [8]:
def generate_answer_with_sources(query: str,
                                retrieved_chunks: List[Tuple[ChunkMetadata, float]]) -> Dict:
    """Generate answer with detailed source attribution using the local LLM."""
    if not retrieved_chunks:
        return {
            'answer': "I couldn't find relevant information to answer your question.",
            'sources': [],
            'confidence': 0.0
        }

    context_parts = []
    sources = []
    for chunk_meta, score in retrieved_chunks:
        context_parts.append(f"[From {chunk_meta.doc_type}, Pages {chunk_meta.page_start}-{chunk_meta.page_end}]")
        context_parts.append(chunk_meta.text)
        context_parts.append("")
        sources.append({
            'doc_type': chunk_meta.doc_type,
            'pages': f"{chunk_meta.page_start}-{chunk_meta.page_end}",
            'relevance': f"{score:.2%}",
            'preview': chunk_meta.text[:100] + "..."
        })
    context = "\n".join(context_parts)

    prompt = f"""You are answering questions about pharmaceutical documentation
including certificates of quality, packaging specifications, and compliance
declarations. Use the provided context to answer the question accurately.
Be specific and cite which document type and pages support your answer.

Context:
{context}

Question: {query}

Instructions:
1. Answer based ONLY on the provided context
2. Mention which document type(s) contain the information
3. Be concise but complete
4. If the context doesn't contain enough information, say so

Answer:"""
    try:
        answer = llm_generate(prompt)
        avg_score = sum(s for _, s in retrieved_chunks) / len(retrieved_chunks)
        return {
            'answer': answer,
            'sources': sources,
            'confidence': avg_score,
            'chunks_used': len(retrieved_chunks)
        }
    except Exception as e:
        import traceback
        print(f"Answer generation error: {type(e).__name__}: {e!r}")
        traceback.print_exc()

        fallback_snippets = "\n\n".join(
            f"[{s['doc_type']}, Pages {s['pages']}]\n{chunk_meta.text}"
            for (chunk_meta, _score), s in zip(retrieved_chunks, sources)
        )
        return {
            'answer': (
                f"The AI answer-generation step failed "
                f"({type(e).__name__}: {str(e) or 'no error message -- see console/traceback for details'}). "
                f"Here are the relevant passages retrieved for your question instead:\n\n"
                f"{fallback_snippets}"
            ),
            'sources': sources,
            'confidence': 0.0
        }

## 🏗️ Enhanced Document Store

In [9]:
# ============================================
# STEP 9: Enhanced Document Store
# ============================================

class EnhancedDocumentStore:
    """
    Manages the complete document processing and retrieval pipeline.
    """

    def __init__(self):
        self.pages_info = []
        self.logical_docs = []
        self.chunks_metadata = []
        self.retriever = IntelligentRetriever()
        self.is_ready = False
        self.processing_stats = {}
        self.filename = None

    def process_pdf(self, pdf_file, filename: str = "document.pdf"):
        """
        Complete PDF processing pipeline.
        """
        self.filename = filename
        self.is_ready = False
        start_time = datetime.now()

        try:
            # Extract and analyze PDF
            self.pages_info, self.logical_docs = extract_and_analyze_pdf(pdf_file)

            # Chunk documents with metadata
            self.chunks_metadata = process_all_documents(self.logical_docs)

            # Build retrieval indices
            self.retriever.build_indices(self.chunks_metadata)

            # Calculate processing statistics
            process_time = (datetime.now() - start_time).total_seconds()
            self.processing_stats = {
                'filename': filename,
                'total_pages': len(self.pages_info),
                'documents_found': len(self.logical_docs),
                'total_chunks': len(self.chunks_metadata),
                'document_types': list(set(doc.doc_type for doc in self.logical_docs)),
                'processing_time': f"{process_time:.1f}s"
            }

            self.is_ready = True
            return True, self.processing_stats

        except Exception as e:
            return False, {'error': str(e)}

    def query(self, question: str, filter_type: Optional[str] = None,
             auto_route: bool = True, k: int = 4) -> Dict:
        """
        Query the document store.
        """
        if not self.is_ready:
            return {
                'answer': "Please upload and process a PDF first.",
                'sources': [],
                'confidence': 0.0
            }

        # Retrieve relevant chunks
        retrieved = self.retriever.retrieve(
            question, k=k,
            filter_doc_type=filter_type,
            auto_route=auto_route
        )

        # Generate answer with sources
        result = generate_answer_with_sources(question, retrieved)
        result['filter_used'] = filter_type or ('auto' if auto_route else 'none')

        return result

    def summarize_all_documents(self) -> Dict:
        """
        Summarize every identified document, not just the top-k most
        similar to a generic "summarize" query.
        """
        if not self.is_ready:
            return {
                'answer': "Please upload and process a PDF first.",
                'sources': [],
                'confidence': 0.0
            }

        if not self.logical_docs:
            return {
                'answer': "No documents were identified in this PDF.",
                'sources': [],
                'confidence': 0.0
            }

        context_parts = []
        sources = []
        for doc in self.logical_docs:
            context_parts.append(
                f"[{doc.doc_type}, Pages {doc.page_start + 1}-{doc.page_end + 1}]"
            )
            context_parts.append(doc.text)
            context_parts.append("")
            sources.append({
                'doc_type': doc.doc_type,
                'pages': f"{doc.page_start + 1}-{doc.page_end + 1}",
                'relevance': "100.00%",  # every document is included, not similarity-ranked
                'preview': doc.text[:100] + "..." if len(doc.text) > 100 else doc.text
            })
        context = "\n".join(context_parts)

        prompt = f"""You are summarizing a set of pharmaceutical documents.
Below is the FULL text of every document found in this PDF ({len(self.logical_docs)}
documents total). Write a clear summary that covers EVERY document listed --
do not skip any of them.

{context}

Instructions:
1. Summarize each of the {len(self.logical_docs)} documents listed above, in order
2. For each, state its document type and the key facts (lot numbers, part
   numbers, dates, key findings, etc. as applicable)
3. Be concise per document, but make sure all {len(self.logical_docs)} are covered

Summary:"""
        try:
            answer = llm_generate(prompt, max_new_tokens=1024)
            return {
                'answer': answer,
                'sources': sources,
                'confidence': 1.0,
                'chunks_used': len(self.logical_docs)
            }
        except Exception as e:
            import traceback
            print(f"Full-document summary error: {type(e).__name__}: {e!r}")
            traceback.print_exc()
            fallback = "\n\n".join(
                f"[{s['doc_type']}, Pages {s['pages']}]\n{doc.text}"
                for doc, s in zip(self.logical_docs, sources)
            )
            return {
                'answer': (
                    f"The AI summary step failed "
                    f"({type(e).__name__}: {str(e) or 'no error message -- see console/traceback for details'}). "
                    f"Here is the full text of all {len(self.logical_docs)} documents instead:\n\n{fallback}"
                ),
                'sources': sources,
                'confidence': 0.0
            }

    def get_document_structure(self) -> List[Dict]:
        """
        Get the document structure for UI display.
        """
        if not self.logical_docs:
            return []

        structure = []
        for doc in self.logical_docs:
            structure.append({
                'id': doc.doc_id,
                'type': doc.doc_type,
                'pages': f"{doc.page_start + 1}-{doc.page_end + 1}",  # 1-indexed for UI
                'chunks': len(doc.chunks) if doc.chunks else 0,
                'preview': doc.text[:200] + "..." if len(doc.text) > 200 else doc.text
            })

        return structure

## 🎨 Gradio Interface with Enhanced Features
Now let's create the sophisticated Gradio interface:

In [19]:
# ============================================
# STEP 10: Gradio Interface
# ============================================
#
# WHAT WE'RE DOING:
# Building the Gradio web UI that ties all pipeline components together.
# The interface has three columns: a PDF upload on the left, document info
# and retrieval settings in the middle, and a chat panel on the right.
# Users upload pharma-blob-sample.pdf, the system processes it, detects
# document boundaries, builds the vector index, and then allows natural
# language Q&A against the identified pharmaceutical sub-documents.
#
# WHY THIS MATTERS:
# The Gradio interface makes the RAG pipeline accessible to non-technical
# users -- a pharmaceutical quality engineer should be able to ask
# "What is the expiration date on the Certificate of Quality?" without
# needing to understand FAISS or embeddings. The document type filter
# dropdown and auto-route toggle give advanced users fine-grained control
# over retrieval scope.
#
# DESIGN DIRECTION:
# Styled like a QC lab document -- a Certificate of Analysis, not a
# generic chat widget. Paper-white panels, hairline dividers, monospace
# data (lot numbers, filenames, page ranges), and a rotated "VERIFIED"
# stamp that appears once a document has been processed, echoing the
# real release stamps this tool's audience reviews all day.
#
# WHAT YOU'LL SEE:
# A browser tab (or inline Colab iframe) showing the full Q&A interface.
# Upload pharma-blob-sample.pdf, wait for processing to complete, then
# type questions in the chat panel on the right.
# ============================================

# Global store instance
doc_store = EnhancedDocumentStore()

# ------------------------------------------------------------------
# Presentation helpers -- turn raw stats/strings into the styled HTML
# fragments used by the status card, the document structure list, and
# the status bar. Kept separate from the pipeline logic above so the
# RAG code stays untouched; these only shape how results are displayed.
# ------------------------------------------------------------------

EMPTY_STATUS_HTML = """
<div class="status-card status-card--empty">
  <div class="status-empty-title">No document loaded</div>
  <div class="status-empty-sub">Upload a pharmaceutical blob PDF on the right to begin.</div>
</div>
"""

def format_status_card(stats):
    """Render processing stats as a certificate-style status card with
    a rotated 'VERIFIED' stamp, mirroring a real Certificate of Quality."""
    return f"""
<div class="status-card status-card--pass">
  <div class="status-stamp">VERIFIED</div>
  <div class="status-row"><span class="status-key">File</span><span class="status-val">{stats['filename']}</span></div>
  <div class="status-row"><span class="status-key">Pages</span><span class="status-val">{stats['total_pages']}</span></div>
  <div class="status-row"><span class="status-key">Documents found</span><span class="status-val">{stats['documents_found']}</span></div>
  <div class="status-row"><span class="status-key">Chunks created</span><span class="status-val">{stats['total_chunks']}</span></div>
  <div class="status-row status-row--wrap"><span class="status-key">Types</span><span class="status-val">{', '.join(stats['document_types'])}</span></div>
  <div class="status-row"><span class="status-key">Processing time</span><span class="status-val">{stats['processing_time']}</span></div>
</div>
"""

def format_status_error(message):
    return f"""
<div class="status-card status-card--error">
  <div class="status-empty-title">Processing failed</div>
  <div class="status-empty-sub">{message}</div>
</div>
"""

def format_doc_structure(structure):
    """Render the detected sub-documents as a list of chip rows instead
    of a plain bullet list."""
    if not structure:
        return ""
    rows = "".join(
        f"""<div class="doc-structure-item">
              <span class="doc-chip">{doc['type']}</span>
              <span class="doc-meta">Pages {doc['pages']} &middot; {doc['chunks']} chunks</span>
            </div>"""
        for doc in structure
    )
    return f'<div class="doc-structure">{rows}</div>'

def format_status_bar(stats=None):
    """Render the footer status bar as a row of pill-shaped stats
    instead of a single line of bold Markdown."""
    ready = stats is not None
    dot_class = "dot--pass" if ready else "dot"
    docs = stats.get('documents_found', 0) if stats else 0
    chunks = stats.get('total_chunks', 0) if stats else 0
    return f"""
<div class="statusbar">
  <span class="stat-pill"><span class="dot {dot_class}"></span>{'Ready' if ready else 'Idle'}</span>
  <span class="stat-pill">Documents <b>{docs}</b></span>
  <span class="stat-pill">Chunks <b>{chunks}</b></span>
</div>
"""

def process_pdf_handler(pdf_file):
    """Handle PDF upload and processing."""
    if pdf_file is None:
        return EMPTY_STATUS_HTML, "", gr.update(choices=["All"])

    # Process the PDF
    success, stats = doc_store.process_pdf(pdf_file,
                                          filename=pdf_file.split('/')[-1] if isinstance(pdf_file, str) else
getattr(pdf_file, 'name', 'pharma-blob-sample.pdf'))

    if success:
        status_msg = format_status_card(stats)
        structure = doc_store.get_document_structure()
        structure_display = format_doc_structure(structure)
        doc_types = ["All"] + stats['document_types']
        return status_msg, structure_display, gr.update(choices=doc_types, value="All")
    else:
        return format_status_error(stats.get('error', 'Unknown error')), "", gr.update(choices=["All"])

def format_sources_block(sources, confidence=None, filter_used=None):
    """Build a collapsible <details> block listing sources (and
    optionally confidence/filter), rendered as HTML inside the
    Markdown chat message so it starts collapsed and expands on click."""
    if not sources:
        return ""

    lines = "".join(
        f"<li>{src['doc_type']} (Pages {src['pages']})"
        + (f" - Relevance: {src['relevance']}" if 'relevance' in src else "")
        + "</li>"
        for src in sources
    )

    footer = ""
    if confidence is not None:
        footer = f'<div class="sources-footer"><em>Confidence: {confidence:.1%} | Filter: {filter_used}</em></div>'

    return (
        '<details class="sources-details">'
        '<summary>Sources</summary>'
        f'<ul>{lines}</ul>'
        f'{footer}'
        '</details>'
    )

def chat_handler(message, history, doc_filter, auto_route, num_chunks):
    """Handle chat interactions."""
    if not doc_store.is_ready:
        response = "Please upload and process a pharmaceutical PDF document first."
        return history + [{"role": "user", "content": message}, {"role": "assistant", "content": response}]

    # Query the document store
    filter_type = None if doc_filter == "All" else doc_filter
    result = doc_store.query(
        message,
        filter_type=filter_type,
        auto_route=auto_route and filter_type is None,
        k=num_chunks
    )

    # Format response, with sources/confidence/filter tucked into a
    # collapsible <details> block so the answer isn't buried under a
    # wall of citation metadata by default.
    response = f"{result['answer']}\n\n"
    response += format_sources_block(result['sources'], result['confidence'], result['filter_used'])

    return history + [{"role": "user", "content": message}, {"role": "assistant", "content": response}]

# ------------------------------------------------------------------
# Theme -- a QC-lab / Certificate-of-Analysis palette rather than a
# generic chat-app theme: paper-white surfaces, deep clinical teal for
# affirmative actions, warm amber reserved for caution states, and a
# monospace face for anything that reads like lab data (filenames,
# lot numbers, page ranges, counts).
# ------------------------------------------------------------------

LAB_THEME = gr.themes.Base(
    font=[gr.themes.GoogleFont("IBM Plex Sans"), "ui-sans-serif", "system-ui", "sans-serif"],
    font_mono=[gr.themes.GoogleFont("IBM Plex Mono"), "ui-monospace", "SFMono-Regular", "monospace"],
).set(
    body_background_fill="#FAFAF7",
    background_fill_primary="#FFFFFF",
    background_fill_secondary="#F1F2ED",
    border_color_primary="#D9DDD5",
    block_background_fill="#FFFFFF",
    block_border_color="#D9DDD5",
    block_border_width="1px",
    block_radius="10px",
    block_label_text_color="#5B6660",
    block_label_text_weight="600",
    block_title_text_color="#1B2420",
    body_text_color="#1B2420",
    body_text_color_subdued="#5B6660",
    button_primary_background_fill="#0E6B57",
    button_primary_background_fill_hover="#0A4A3C",
    button_primary_text_color="#FFFFFF",
    button_primary_border_color="#0E6B57",
    button_secondary_background_fill="#FFFFFF",
    button_secondary_background_fill_hover="#F1F2ED",
    button_secondary_border_color="#D9DDD5",
    button_secondary_text_color="#1B2420",
    input_background_fill="#FFFFFF",
    input_border_color="#D9DDD5",
    input_border_color_focus="#0E6B57",
    slider_color="#0E6B57",
    checkbox_background_color_selected="#0E6B57",
    checkbox_border_color_selected="#0E6B57",
    shadow_drop="0 1px 2px rgba(27,36,32,0.06)",
    # ---- Dark-mode variants pinned to the SAME light values. ----
    # Without these, components like File and Chatbot fall back to
    # Gradio's own built-in dark palette (and default font) whenever
    # the visitor's OS/browser is in dark mode, which is why the
    # upload box and chat panel were rendering as plain black boxes
    # in a different typeface. Pinning every *_dark token stops the
    # theme from switching at all.
    body_background_fill_dark="#FAFAF7",
    background_fill_primary_dark="#FFFFFF",
    background_fill_secondary_dark="#F1F2ED",
    border_color_primary_dark="#D9DDD5",
    block_background_fill_dark="#FFFFFF",
    block_border_color_dark="#D9DDD5",
    block_label_text_color_dark="#5B6660",
    block_title_text_color_dark="#1B2420",
    body_text_color_dark="#1B2420",
    body_text_color_subdued_dark="#5B6660",
    button_primary_background_fill_dark="#0E6B57",
    button_primary_background_fill_hover_dark="#0A4A3C",
    button_primary_text_color_dark="#FFFFFF",
    button_primary_border_color_dark="#0E6B57",
    button_secondary_background_fill_dark="#FFFFFF",
    button_secondary_background_fill_hover_dark="#F1F2ED",
    button_secondary_border_color_dark="#D9DDD5",
    button_secondary_text_color_dark="#1B2420",
    input_background_fill_dark="#FFFFFF",
    input_border_color_dark="#D9DDD5",
    input_border_color_focus_dark="#0E6B57",
    checkbox_background_color_selected_dark="#0E6B57",
    checkbox_border_color_selected_dark="#0E6B57",
)

COMPACT_CSS = """
:root {
    --paper: #FAFAF7;
    --paper-2: #F1F2ED;
    --ink: #1B2420;
    --ink-soft: #5B6660;
    --line: #D9DDD5;
    --teal: #0E6B57;
    --teal-dark: #0A4A3C;
    --teal-tint: #EAF5F1;
    --amber: #92621C;
    --amber-tint: #FBF1DF;
    --steel: #2F4B6E;
    --steel-tint: #EAF0F7;
}

.gradio-container { max-width: 100% !important; background: var(--paper) !important; }
#left_panel { font-size: 0.88em; }

/* Belt-and-suspenders: if the visitor's browser is in dark mode,
   Gradio scopes a `.dark` class onto the app root and reads its own
   CSS custom properties from it. Re-pin those to the same light
   values here so no component (including ones added by future
   Gradio versions) can silently switch palettes. */
.dark, .gradio-container.dark {
    --body-background-fill: var(--paper) !important;
    --background-fill-primary: #FFFFFF !important;
    --background-fill-secondary: var(--paper-2) !important;
    --border-color-primary: var(--line) !important;
    --block-background-fill: #FFFFFF !important;
    --block-border-color: var(--line) !important;
    --block-label-text-color: var(--ink-soft) !important;
    --block-title-text-color: var(--ink) !important;
    --body-text-color: var(--ink) !important;
    --body-text-color-subdued: var(--ink-soft) !important;
    --input-background-fill: #FFFFFF !important;
    --input-border-color: var(--line) !important;
    --button-secondary-background-fill: #FFFFFF !important;
    --button-secondary-border-color: var(--line) !important;
    --button-secondary-text-color: var(--ink) !important;
}

/* Force the type family everywhere, including inside components
   (File dropzone, Chatbot) whose internal markup sits below the
   selectors above and previously kept the browser's default font. */
.gradio-container, .gradio-container * {
    font-family: 'IBM Plex Sans', ui-sans-serif, system-ui, sans-serif !important;
}
.status-val, .doc-chip, .doc-meta, .stat-pill, .hero-eyebrow,
.sources-details .sources-footer, code, pre {
    font-family: 'IBM Plex Mono', ui-monospace, monospace !important;
}

/* -------------------- Hero / document header -------------------- */
.hero { padding: 2px 2px 16px 2px; border-bottom: 1px solid var(--line); margin-bottom: 16px; }
.hero-eyebrow {
    font-family: 'IBM Plex Mono', monospace; font-size: 0.72em; letter-spacing: 0.14em;
    text-transform: uppercase; color: var(--teal); margin-bottom: 8px;
}
.hero-title { font-size: 1.55em; font-weight: 650; margin: 0 0 6px 0; color: var(--ink); letter-spacing: -0.01em; }
.hero-sub { font-size: 0.92em; color: var(--ink-soft); margin: 0; max-width: 660px; line-height: 1.5; }

/* -------------------- Section labels (panel headers) -------------------- */
#left_panel h3, #right_panel h3 {
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 0.72em !important;
    letter-spacing: 0.12em !important;
    text-transform: uppercase !important;
    color: var(--ink-soft) !important;
    font-weight: 600 !important;
    border-bottom: 1px solid var(--line) !important;
    padding-bottom: 6px !important;
    margin: 14px 0 10px 0 !important;
}

.gr-button, button { padding: 4px 10px !important; }

/* Dropzone: compact height, wide horizontal layout, readable text size.
   Explicit background/text colors here (not just on .dark above)
   because the File component paints its own surface regardless of
   the light/dark class present on the root. */
#pdf_upload, #pdf_upload * {
    background: #FFFFFF !important;
    color: var(--ink) !important;
    border-color: var(--line) !important;
}
#pdf_upload svg { color: var(--teal) !important; fill: currentColor !important; }
#pdf_upload { max-height: 130px !important; border-color: var(--line) !important; }
#pdf_upload .wrap {
    min-height: 90px !important; height: 90px !important;
    display: flex !important; flex-direction: row !important;
    align-items: center !important; justify-content: center !important;
    gap: 10px !important; flex-wrap: wrap !important;
}
#pdf_upload .wrap > * {
    display: flex !important; flex-direction: row !important;
    align-items: center !important; gap: 6px !important;
}
#pdf_upload .wrap, #pdf_upload .wrap * {
    font-size: 0.95em !important; line-height: 1.2 !important;
}
#pdf_upload .wrap svg { width: 20px !important; height: 20px !important; color: var(--teal) !important; }

/* Left panel: stretch to match right column without stray gaps
   between children (see original note: children default to
   flex-grow:1, which left empty space inside each block before a
   file is uploaded). Forcing flex: 0 0 auto packs everything from
   the top. */
#left_panel {
    display: flex !important; flex-direction: column !important;
    justify-content: flex-start !important; gap: 4px !important;
}
#left_panel > * { flex: 0 0 auto !important; }

/* Message textbox + Send button aligned in one row. */
#ask_row { display: flex !important; align-items: center !important; }
#ask_row > * { align-self: center !important; }

/* -------------------- Status card (Certificate-of-Analysis style) -------------------- */
.status-card {
    position: relative;
    background: var(--paper) !important;
    border: 1px solid var(--line);
    border-radius: 10px;
    padding: 14px 16px;
}
.status-card--pass {
    border-color: #BFE3D6;
    background: linear-gradient(180deg, var(--teal-tint), var(--paper) 65%) !important;
    padding-right: 84px;
}
.status-card--error { border-color: #E4C7AE; background: var(--amber-tint) !important; }
.status-card--empty { border-style: dashed; }
.status-empty-title { font-weight: 600; color: var(--ink); font-size: 0.92em; }
.status-empty-sub { color: var(--ink-soft); font-size: 0.85em; margin-top: 3px; }

.status-stamp {
    position: absolute; top: 10px; right: 12px;
    width: 60px; height: 60px; border-radius: 50%;
    border: 2px solid var(--teal); color: var(--teal);
    display: flex; align-items: center; justify-content: center; text-align: center;
    font-family: 'IBM Plex Mono', monospace; font-size: 0.6em; font-weight: 700; letter-spacing: 0.03em;
    transform: rotate(-9deg); opacity: 0.9;
}
.status-stamp::after {
    content: ""; position: absolute; inset: 4px; border: 1px solid var(--teal); border-radius: 50%;
}

.status-row {
    display: flex; justify-content: space-between; align-items: baseline;
    gap: 14px; padding: 4px 0; font-size: 0.85em;
    border-bottom: 1px dashed var(--line);
}
.status-row:last-child { border-bottom: none; }
.status-key {
    color: var(--ink-soft);
    white-space: nowrap;   /* keep the label on one line, however long the value is */
    flex-shrink: 0;
}
.status-val {
    font-family: 'IBM Plex Mono', monospace !important;
    color: var(--ink);
    text-align: right;
    word-break: break-word;
}
/* Long, comma-separated values (e.g. the document-type list) read
   better left-aligned and top-aligned once they wrap to several
   lines, rather than ragged-right under a baseline-aligned label. */
.status-row--wrap { align-items: flex-start; }
.status-row--wrap .status-val { text-align: left; }

/* -------------------- Document structure list -------------------- */
.doc-structure { display: flex; flex-direction: column; gap: 6px; margin-top: 8px; }
.doc-structure-item {
    display: flex; align-items: center; gap: 8px; flex-wrap: wrap;
    font-size: 0.83em; padding: 6px 8px;
    background: var(--paper-2); border-radius: 8px; border: 1px solid var(--line);
}
.doc-chip {
    font-family: 'IBM Plex Mono', monospace; font-size: 0.68em; padding: 2px 7px;
    border-radius: 999px; background: var(--steel-tint); color: var(--steel);
    white-space: nowrap; text-transform: uppercase; letter-spacing: 0.03em;
}
.doc-meta { color: var(--ink-soft); }

/* -------------------- Status bar (footer) -------------------- */
.statusbar { display: flex; gap: 10px; align-items: center; flex-wrap: wrap; padding: 6px 2px; }
.stat-pill {
    font-family: 'IBM Plex Mono', monospace; font-size: 0.78em; color: var(--ink);
    background: var(--paper-2); border: 1px solid var(--line);
    padding: 4px 10px; border-radius: 999px; display: inline-flex; align-items: center; gap: 6px;
}
.stat-pill b { color: var(--teal); font-weight: 700; }
.dot { width: 7px; height: 7px; border-radius: 50%; background: var(--ink-soft); display: inline-block; }
.dot--pass { background: var(--teal); }

/* -------------------- Chip-style utility buttons -------------------- */
.chip-btn, .chip-btn button {
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 0.78em !important;
    border-radius: 999px !important;
    letter-spacing: 0.01em;
}

/* -------------------- Collapsible sources block inside chat -------------------- */
.sources-details {
    margin-top: 6px;
    border: 1px solid var(--line);
    border-radius: 8px;
    padding: 4px 8px;
    background: var(--paper-2);
}
.sources-details summary {
    cursor: pointer;
    font-weight: 600;
    color: var(--teal-dark);
    padding: 4px 0;
    list-style: revert;
    font-size: 0.9em;
}
.sources-details summary:hover { opacity: 0.8; }
.sources-details ul { margin: 6px 0 2px 0; padding-left: 20px; }
.sources-details .sources-footer {
    margin-top: 4px;
    opacity: 0.75;
    font-size: 0.85em;
    font-family: 'IBM Plex Mono', monospace;
}

/* -------------------- Chatbot -------------------- */
/* Same reasoning as #pdf_upload above: force every descendant, not
   just the outer frame, so the empty-state canvas and message
   bubbles can't fall back to Gradio's dark surface. */
#chatbot, #chatbot * {
    background: var(--paper) !important;
    color: var(--ink) !important;
    border-color: var(--line) !important;
}
#chatbot { border: 1px solid var(--line) !important; }
#chatbot .message.user, #chatbot [data-testid="user"] {
    background: var(--teal-tint) !important;
}
#chatbot .message.bot, #chatbot [data-testid="bot"] {
    background: #FFFFFF !important;
    border: 1px solid var(--line) !important;
}
"""

# ------------------------------------------------------------------
# BUG FIX: the "share" icon on gr.Chatbot is a *built-in* Gradio
# feature (Chatbot's `buttons` param, default
# ["share", "copy", "copy_all"]) whose only job is to upload the
# conversation to a Hugging Face Spaces Discussion thread. This app
# runs from Colab via a temporary gradio.live tunnel, not a real HF
# Space, so there is nowhere for it to post to -- the click handler
# fires and silently no-ops. Rather than fight the rendered DOM with a
# CSS/JS MutationObserver hack (which was unreliable), just don't ask
# Gradio to render the button: pass `buttons=["copy", "copy_all"]`
# (omitting "share") to gr.Chatbot below. This keeps the useful
# per-message copy button and the copy-all button, and removes the
# dead share button at the source instead of hiding it after the fact.
# ------------------------------------------------------------------

def create_interface():
    """Create the Gradio interface for pharmaceutical document Q&A."""

    with gr.Blocks() as demo:
        gr.HTML("""
        <div class="hero">
            <div class="hero-eyebrow">RAG-Assisted Review &middot; Local Inference Only</div>
            <h1 class="hero-title">Pharmaceutical Document Q&amp;A</h1>
            <p class="hero-sub">Upload a pharmaceutical blob PDF (e.g. pharma-blob-sample.pdf) to identify
            document types, build a searchable index, and ask questions in natural language.</p>
        </div>
        """)

        with gr.Row(equal_height=True):
            # Left - Document info and settings
            with gr.Column(scale=1, elem_id="left_panel"):
                gr.Markdown("### Document Info")
                status_output = gr.Markdown(value=EMPTY_STATUS_HTML)
                structure_output = gr.Markdown(value="", label="Document Structure")

                gr.Markdown("### Retrieval Settings")

                doc_filter = gr.Dropdown(
                    choices=["All"],
                    value="All",
                    label="Document Type Filter",
                    info="Filter search to a specific pharmaceutical document type"
                )

                auto_route = gr.Checkbox(
                    value=True,
                    label="Auto-Route Queries",
                    info="Automatically detect the most relevant document type"
                )

                num_chunks = gr.Slider(
                    minimum=1,
                    maximum=10,
                    value=4,
                    step=1,
                    label="Chunks to Retrieve"
                )

            # Right - Upload (small, on top) + Chat (fills remaining height, below)
            with gr.Column(scale=4, elem_id="right_panel"):
                with gr.Group():
                    pdf_input = gr.File(
                        label="Upload Pharmaceutical PDF",
                        file_types=[".pdf"],
                        type="filepath",
                        elem_id="pdf_upload"
                    )

                    with gr.Row():
                        process_btn = gr.Button(
                            "Process Document",
                            variant="primary",
                            size="sm",
                            scale=2
                        )
                        clear_all_btn = gr.Button(
                            "Clear All",
                            variant="secondary",
                            size="sm",
                            scale=1
                        )

                gr.Markdown("### Ask Questions")
                chatbot = gr.Chatbot(
                    label="Conversation",
                    height=560,
                    elem_id="chatbot",
                    show_label=False,
                    buttons=["copy", "copy_all"],  # omit "share": it only
                    # works on a real HF Space (posts to a Spaces
                    # Discussion thread); on a Colab/gradio.live tunnel
                    # it has nowhere to send the conversation, so the
                    # click silently does nothing.
                )

                with gr.Row(elem_id="ask_row"):
                    msg_input = gr.Textbox(
                        label="Ask a question",
                        placeholder="e.g., What is the lot number? What sterilization method was used?",
                        scale=4,
                        show_label=False
                    )
                    send_btn = gr.Button("Send", scale=1, variant="primary")

                with gr.Row():
                    clear_chat_btn = gr.Button("Clear Chat", size="sm", scale=1, elem_classes=["chip-btn"])
                    example_btn1 = gr.Button("Summarize Document", size="sm", scale=1, elem_classes=["chip-btn"])
                    example_btn2 = gr.Button("Find Lot Numbers", size="sm", scale=1, elem_classes=["chip-btn"])

        # Status bar at the bottom
        with gr.Row():
            status_bar = gr.Markdown(value=format_status_bar(), elem_id="status_bar")

        # Event handlers
        def update_status_bar():
            """Update the status bar with current statistics."""
            if doc_store.is_ready:
                return format_status_bar(doc_store.processing_stats)
            return format_status_bar()

        def clear_all():
            """Clear everything and reset the interface."""
            global doc_store
            doc_store = EnhancedDocumentStore()
            return (
                None,  # pdf_input
                EMPTY_STATUS_HTML,  # status_output
                "",  # structure_output
                gr.update(choices=["All"], value="All"),  # doc_filter
                [],  # chatbot
                "",  # msg_input
            )

        # Example question handlers
        def ask_summary(history):
            result = doc_store.summarize_all_documents()
            response = f"{result['answer']}\n\n"
            response += format_sources_block(result['sources'])
            message = "Can you provide a summary of the main points in this document?"
            return history + [
                {"role": "user", "content": message},
                {"role": "assistant", "content": response}
            ]

        def ask_lot_numbers(history):
            return chat_handler(
                "What lot numbers or batch numbers are mentioned in these documents?",
                history, doc_filter.value, auto_route.value, num_chunks.value
            )

        # Wire up all the events.
        #
        # LOADING-BOX FIX: Gradio shows a pending overlay (spinner +
        # "x.x/xx.xs" eta text) on every OUTPUT component of a running
        # event. status_bar is a thin, mostly-empty row pinned to the
        # bottom of the page, so bundling it into the same event as
        # chatbot/status_output made that overlay render as its own
        # floating box down there instead of staying inside the chat
        # panel. Fix: update status_bar in a separate chained step with
        # show_progress="hidden", so only the chatbot (or the status
        # card, for PDF processing) shows a loading indicator.
        process_btn.click(
            fn=process_pdf_handler,
            inputs=[pdf_input],
            outputs=[status_output, structure_output, doc_filter]
        ).then(
            fn=update_status_bar,
            outputs=[status_bar],
            show_progress="hidden"
        )

        clear_all_btn.click(
            fn=clear_all,
            outputs=[pdf_input, status_output, structure_output, doc_filter,
                    chatbot, msg_input]
        ).then(
            fn=update_status_bar,
            outputs=[status_bar],
            show_progress="hidden"
        )

        # Chat interactions
        msg_input.submit(
            fn=chat_handler,
            inputs=[msg_input, chatbot, doc_filter, auto_route, num_chunks],
            outputs=[chatbot]
        ).then(
            fn=update_status_bar,
            outputs=[status_bar],
            show_progress="hidden"
        ).then(
            lambda: "",
            outputs=[msg_input],
            show_progress="hidden"
        )

        send_btn.click(
            fn=chat_handler,
            inputs=[msg_input, chatbot, doc_filter, auto_route, num_chunks],
            outputs=[chatbot]
        ).then(
            fn=update_status_bar,
            outputs=[status_bar],
            show_progress="hidden"
        ).then(
            lambda: "",
            outputs=[msg_input],
            show_progress="hidden"
        )

        clear_chat_btn.click(
            lambda: [],
            outputs=[chatbot]
        )

        example_btn1.click(
            fn=ask_summary,
            inputs=[chatbot],
            outputs=[chatbot]
        ).then(
            fn=update_status_bar,
            outputs=[status_bar],
            show_progress="hidden"
        )

        example_btn2.click(
            fn=ask_lot_numbers,
            inputs=[chatbot],
            outputs=[chatbot]
        ).then(
            fn=update_status_bar,
            outputs=[status_bar],
            show_progress="hidden"
        )

        # Auto-process when PDF is uploaded
        pdf_input.change(
            fn=process_pdf_handler,
            inputs=[pdf_input],
            outputs=[status_output, structure_output, doc_filter]
        ).then(
            fn=update_status_bar,
            outputs=[status_bar],
            show_progress="hidden"
        )

    return demo

In [20]:
# ============================================
# STEP 11: Launch the Application
# ============================================
#
# WHAT WE'RE DOING:
# Creating the Gradio interface instance and launching it. The share=True
# flag generates a public temporary URL so anyone with the link can access
# the running app from outside Colab. debug=True prints server-side logs
# to the cell output so you can see what happens when the PDF is processed
# and when questions are asked.
#
# WHY THIS MATTERS:
# This is the entry point that makes everything visible and interactive.
# Without this cell, all the pipeline code above would be defined but
# nothing would be rendered for the user. Once launched, the app stays
# alive until you stop the cell or the Colab session times out.
#
# WHAT YOU'LL SEE:
# A public Gradio URL (https://....gradio.live) and an inline iframe
# showing the Pharmaceutical Document Q&A System. Open the URL or use the
# iframe, upload pharma-blob-sample.pdf, and start asking questions.
# ============================================

demo = create_interface()
demo.launch(share=True, debug=True, theme=LAB_THEME, css=COMPACT_CSS)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://400807498fa1ab5a82.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Starting PDF extraction and analysis...
Extracted 10 pages
Analyzing document structure...
  Page 0: New document detected - Cover Letter
  Page 1: New document detected - Certificate Of Quality
  Page 2: New document detected - Certificate Of Quality
  Page 3: New document detected - Packaging Specification
  Page 5: New document detected - BSE/TSE Declaration
  Page 6: New document detected - Material Description
  Page 7: New document detected - Supplier Qualification
  Page 9: New document detected - Chain Of Custody
Identified 8 logical documents
   - Cover Letter: Pages 0-0
   - Certificate Of Quality: Pages 1-1
   - Certificate Of Quality: Pages 2-2
   - Packaging Specification: Pages 3-4
   - BSE/TSE Declaration: Pages 5-5
   - Material Description: Pages 6-6
   - Supplier Qualification: Pages 7-8
   - Chain Of Custody: Pages 9-9
  Cover Letter: Created 3 chunks
  Certificate Of Quality: Created 2 chunks
  Certificate Of Quality: Created 2 chunks
  Packaging Specification: Crea

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Indexed 19 chunks across 7 document types
Query routed to: Certificate Of Quality (confidence: 0.95)
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://400807498fa1ab5a82.gradio.live
